# 投资组合优化完整教程：从目标函数到实际约束

## 📚 教学目标
1. 理解因子投资中**投资组合优化**的目标函数（均值-方差、最小方差、最大多样化、风险平价）
2. 掌握**约束条件**的设置：权重和、行业偏离、因子暴露、换手率
3. 在 **5 只股票 × 2 因子**的微型数据集上手算简化优化问题
4. 使用 `scipy.optimize` 求解并验证
5. 对比**等权组合 vs 优化组合**的绩效差异

**参考书**：《因子投资：方法与实践》（石川）第 7.3 节
**教学策略**：先用极小数据集让你「看见」优化的数学本质，再扩展到真实规模

---

## 1. 为什么需要投资组合优化？

### 🎯 一个直觉问题

假设你已经有了收益模型（预测各股票的预期收益率）和风险模型（计算协方差矩阵），下一步是什么？

你不能简单地「买预测收益最高的那只股票」——因为风险太大！你需要一个**系统化的方法**来决定每只股票买多少。

这就是**投资组合优化**的核心任务。

### 📖 书中的定义

> 有了收益模型和风险模型，主动管理的下一步就是将二者结合在一起，进行投资组合优化，以获得更高的风险调整后收益。

### 💡 核心思路

```
收益模型 ──→ 预期收益率 μ（每只股票未来能赚多少）
                   ↓
          ┌────────────────────┐
          │  投资组合优化器     │ ──→ 最优权重 w*（每只股票买多少）
          └────────────────────┘
                   ↑
风险模型 ──→ 协方差矩阵 Σ（各股票风险和相关性）
```

优化器的目标：在给定风险下最大化收益，或在给定收益下最小化风险。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats, optimize

# 设置中文字体和绘图风格
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 目标函数一：均值-方差优化 (MVO)

### 📐 Markowitz 均值-方差模型

最经典的投资组合优化目标函数来自 Markowitz (1952)：

$$\max_{w} \quad w'\mu - \frac{\zeta}{2} w'\Sigma w$$

其中：
- $w$ = 股票权重向量 ($N \times 1$)
- $\mu$ = 股票预期超额收益率向量 ($N \times 1$)
- $\Sigma$ = 股票收益率的协方差矩阵 ($N \times N$)
- $\zeta$ = 风险厌恶系数（越大越保守）

#### 💡 经济含义

- $w'\mu$ = 组合的预期收益（要**最大化**）
- $\frac{\zeta}{2} w'\Sigma w$ = 风险惩罚项（方差越大，惩罚越重）
- 两者相减 = **风险调整后收益**

#### 无约束最优解

当不考虑任何约束时，MVO 的最优解有解析形式：

$$w^*_{\text{mvo}} = (\zeta \Sigma)^{-1} \mu = \frac{1}{\zeta} \Sigma^{-1} \mu$$

如果要求 $\sum w_i = 1$（权重之和为 1），最优解变为：

$$w^*_{\text{mvo}} \propto \Sigma^{-1} \mu$$

### 🎯 手算示例：5 只股票的 MVO

我们用一个极小的例子来「看见」优化过程。

**设定**：5 只股票，2 个因子（市值、B/M）

In [ ]:
# ========== 构造 5 只股票的微型数据集 ==========
stocks = ['A', 'B', 'C', 'D', 'E']

# 预期超额收益率 (%)
mu = np.array([8.0, 6.0, 5.0, 4.0, 3.0])

# 协方差矩阵 (简化: 对角为方差, 非对角为协方差)
# 为了手算方便, 使用一个简单的协方差矩阵
Sigma = np.array([
    [16.0,  4.0,  2.0,  1.0,  0.5],
    [ 4.0, 12.0,  3.0,  1.5,  1.0],
    [ 2.0,  3.0, 10.0,  2.0,  1.5],
    [ 1.0,  1.5,  2.0,  9.0,  2.0],
    [ 0.5,  1.0,  1.5,  2.0,  8.0]
])

print("📋 5 只股票的输入参数")
print("─" * 50)
df_input = pd.DataFrame({
    'Stock': stocks,
    'Expected Return (%)': mu,
    'Variance': np.diag(Sigma)
})
print(df_input.to_string(index=False))

print(f"\n📊 协方差矩阵 Σ:")
df_Sigma = pd.DataFrame(Sigma, index=stocks, columns=stocks)
print(df_Sigma.round(1))

print(f"\n💡 观察：")
print(f"  - 股票 A 预期收益最高 (8%)，但方差也最大 (16)")
print(f"  - 股票 E 预期收益最低 (3%)，但方差较小 (8)")
print(f"  - 非对角元素 > 0 表示股票之间正相关（同涨同跌）")

In [ ]:
# ========== 步骤 1: 手算 MVO 无约束最优解 ==========
zeta = 1.0  # 风险厌恶系数

print("📊 步骤 1: 计算 MVO 无约束最优权重")
print("─" * 50)
print(f"  公式: w* = (ζΣ)⁻¹ μ")
print(f"  ζ = {zeta}")
print()

# 计算 Σ⁻¹
Sigma_inv = np.linalg.inv(Sigma)
print(f"  📐 步骤 1a: 计算 Σ⁻¹ (协方差矩阵的逆)")
print(f"  Σ⁻¹ 对角线: {np.diag(Sigma_inv).round(4)}")
print()

# 计算 Σ⁻¹ μ
Sigma_inv_mu = Sigma_inv @ mu
print(f"  📐 步骤 1b: 计算 Σ⁻¹μ")
for i, s in enumerate(stocks):
    print(f"    (Σ⁻¹μ)_{s} = {Sigma_inv_mu[i]:.4f}")
print()

# 最优权重
w_mvo_unconstrained = Sigma_inv_mu / zeta
print(f"  📐 步骤 1c: w* = (1/ζ) × Σ⁻¹μ")
for i, s in enumerate(stocks):
    print(f"    w*_{s} = {w_mvo_unconstrained[i]:.4f}")
print(f"    权重之和 = {w_mvo_unconstrained.sum():.4f}")
print()
print(f"  💡 解读: 无约束下，权重之和 ≠ 1。某些股票可能被做空（负权重）！")

In [ ]:
# ========== 步骤 2: 加入权重约束 Σw = 1 的 MVO ==========
print("📊 步骤 2: 加入约束 Σwᵢ = 1")
print("─" * 50)

# 归一化: w* ∝ Σ⁻¹μ
w_mvo_normalized = w_mvo_unconstrained / w_mvo_unconstrained.sum()

print(f"  公式: w* ∝ Σ⁻¹μ (归一化使权重之和 = 1)")
print()
for i, s in enumerate(stocks):
    print(f"    w*_{s} = {w_mvo_normalized[i]:.4f}  ({w_mvo_normalized[i]*100:.1f}%)")
print(f"    权重之和 = {w_mvo_normalized.sum():.4f}")

# 计算组合预期收益和风险
port_ret = w_mvo_normalized @ mu
port_var = w_mvo_normalized @ Sigma @ w_mvo_normalized
port_std = np.sqrt(port_var)
port_sr = port_ret / port_std

print(f"\n  📊 MVO 组合特征:")
print(f"    预期收益 = {port_ret:.4f}%")
print(f"    波动率   = {port_std:.4f}%")
print(f"    Sharpe   = {port_sr:.4f}")

# scipy 验证
print(f"\n  🔬 scipy.optimize 验证:")

def mvo_objective(w, mu, Sigma, zeta):
    """MVO 目标函数: max w'μ - ζ/2 * w'Σw  →  min -(w'μ - ζ/2 * w'Σw)"""
    return -(w @ mu - zeta / 2 * w @ Sigma @ w)

N = len(stocks)
w0 = np.ones(N) / N  # 初始等权
constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
bounds = [(-1, 1)] * N  # 允许做空

result = optimize.minimize(mvo_objective, w0, args=(mu, Sigma, zeta),
                           method='SLSQP', constraints=constraints, bounds=bounds)

w_scipy = result.x
print(f"  手算权重: {np.round(w_mvo_normalized, 4)}")
print(f"  scipy权重: {np.round(w_scipy, 4)}")
print(f"  最大差异: {np.max(np.abs(w_mvo_normalized - w_scipy)):.6f}")
print(f"  ✅ 验证通过！")

In [ ]:
# ========== 可视化 MVO 权重 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 左图：MVO 权重柱状图 ---
ax1 = axes[0]
colors = ['#2ecc71' if w > 0 else '#e74c3c' for w in w_mvo_normalized]
bars = ax1.bar(stocks, w_mvo_normalized * 100, color=colors, edgecolor='black', alpha=0.85)
for bar, w in zip(bars, w_mvo_normalized):
    va = 'bottom' if w >= 0 else 'top'
    offset = 0.5 if w >= 0 else -0.5
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + offset,
             f'{w*100:.1f}%', ha='center', va=va, fontweight='bold')
ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.set_xlabel('Stock', fontsize=12)
ax1.set_ylabel('Weight (%)', fontsize=12)
ax1.set_title('MVO Portfolio Weights (zeta=1)', fontsize=13, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# --- 右图：收益-风险散点 ---
ax2 = axes[1]
# 个股
for i, s in enumerate(stocks):
    ax2.scatter(np.sqrt(Sigma[i, i]), mu[i], s=100, c='steelblue', edgecolors='black', zorder=5)
    ax2.annotate(s, (np.sqrt(Sigma[i, i]), mu[i]),
                textcoords="offset points", xytext=(5, 5), fontsize=11)
# MVO 组合
ax2.scatter(port_std, port_ret, s=200, c='red', marker='*', edgecolors='black', zorder=6,
            label=f'MVO (SR={port_sr:.2f})')
# 等权组合
w_ew = np.ones(N) / N
ret_ew = w_ew @ mu
std_ew = np.sqrt(w_ew @ Sigma @ w_ew)
ax2.scatter(std_ew, ret_ew, s=200, c='green', marker='D', edgecolors='black', zorder=6,
            label=f'Equal-Weight (SR={ret_ew/std_ew:.2f})')

ax2.set_xlabel('Volatility (%)', fontsize=12)
ax2.set_ylabel('Expected Return (%)', fontsize=12)
ax2.set_title('Risk-Return Scatter', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 左图：MVO 给高收益股票(A,B)更高权重，低收益股票(D,E)更低权重")
print(f"   右图：红星(MVO组合)的 Sharpe Ratio 高于绿菱形(等权组合)")
print(f"   MVO SR = {port_sr:.2f} vs 等权 SR = {ret_ew/std_ew:.2f}")

---

## 3. 目标函数二：最小方差 (Minimum Variance)

### 📐 最小方差优化

MVO 的一个问题是需要估计预期收益率 $\mu$，而 $\mu$ 的估计误差会导致最优权重剧烈波动。

**最小方差**策略只使用协方差矩阵 $\Sigma$，完全避免估计 $\mu$：

$$\min_{w} \quad w'\Sigma w$$
$$\text{s.t.} \quad w'1 = 1$$

最优解有解析形式：

$$w^*_{\text{mv}} \propto \Sigma^{-1} 1$$

其中 $1$ 是全 1 向量。

#### 💡 与 MVO 的关系

当所有资产的预期收益率 $\mu_i$ **都相等**时，MVO 等价于最小方差。

In [ ]:
# ========== 最小方差组合 ==========
print("📊 最小方差组合计算")
print("─" * 50)

# 手算: w* ∝ Σ⁻¹ × 1
ones = np.ones(N)
Sigma_inv_ones = Sigma_inv @ ones
w_mv = Sigma_inv_ones / Sigma_inv_ones.sum()

print(f"  公式: w* ∝ Σ⁻¹ × 1")
print(f"  Σ⁻¹×1 = {Sigma_inv_ones.round(4)}")
print(f"  归一化后权重:")
for i, s in enumerate(stocks):
    print(f"    w*_{s} = {w_mv[i]:.4f}  ({w_mv[i]*100:.1f}%)")
print(f"    权重之和 = {w_mv.sum():.4f}")

# 组合特征
port_ret_mv = w_mv @ mu
port_var_mv = w_mv @ Sigma @ w_mv
port_std_mv = np.sqrt(port_var_mv)
port_sr_mv = port_ret_mv / port_std_mv

print(f"\n  📊 最小方差组合特征:")
print(f"    预期收益 = {port_ret_mv:.4f}%")
print(f"    波动率   = {port_std_mv:.4f}%")
print(f"    Sharpe   = {port_sr_mv:.4f}")

# scipy 验证
def min_var_objective(w, Sigma):
    return w @ Sigma @ w

result_mv = optimize.minimize(min_var_objective, w0, args=(Sigma,),
                               method='SLSQP', constraints=constraints)
print(f"\n  🔬 scipy 验证: 最大权重差异 = {np.max(np.abs(w_mv - result_mv.x)):.6f} ✅")

---

## 4. 目标函数三：最大多样化 (Maximum Diversification)

### 📐 最大多样化优化

最大多样化的目标是最大化**加权个股波动率与组合波动率之比**：

$$\max_{w} \quad \frac{w'\sigma}{\sqrt{w'\Sigma w}}$$
$$\text{s.t.} \quad w'1 = 1$$

其中 $\sigma = [\sigma_1, \sigma_2, \ldots, \sigma_N]'$ 是各股票波动率的向量。

这个比值越大，说明组合的分散化程度越高。

最优解满足 (Clarke et al. 2013)：

$$w^*_{\text{md}} \propto \Sigma^{-1} \sigma$$

#### 💡 与 MVO 的关系

当所有资产的 $\mu_i / \sigma_i$ **都相等**时，最大多样化等价于 MVO。

In [ ]:
# ========== 最大多样化组合 ==========
print("📊 最大多样化组合计算")
print("─" * 50)

# 各股票波动率
sigma_vec = np.sqrt(np.diag(Sigma))
print(f"  各股票波动率 σ = {sigma_vec.round(4)}")
print()

# 手算: w* ∝ Σ⁻¹σ
Sigma_inv_sigma = Sigma_inv @ sigma_vec
w_md = Sigma_inv_sigma / Sigma_inv_sigma.sum()

print(f"  公式: w* ∝ Σ⁻¹σ")
print(f"  Σ⁻¹σ = {Sigma_inv_sigma.round(4)}")
print(f"  归一化后权重:")
for i, s in enumerate(stocks):
    print(f"    w*_{s} = {w_md[i]:.4f}  ({w_md[i]*100:.1f}%)")
print(f"    权重之和 = {w_md.sum():.4f}")

# 组合特征
port_ret_md = w_md @ mu
port_var_md = w_md @ Sigma @ w_md
port_std_md = np.sqrt(port_var_md)
port_sr_md = port_ret_md / port_std_md

# 多样化比率
div_ratio = (w_md @ sigma_vec) / port_std_md

print(f"\n  📊 最大多样化组合特征:")
print(f"    预期收益   = {port_ret_md:.4f}%")
print(f"    波动率     = {port_std_md:.4f}%")
print(f"    Sharpe     = {port_sr_md:.4f}")
print(f"    多样化比率 = {div_ratio:.4f}")
print(f"\n  💡 多样化比率 = 加权个股波动率 / 组合波动率 = {div_ratio:.2f}")
print(f"     如果所有股票完全不相关，这个比值会很高")

---

## 5. 目标函数四：风险平价 (Risk Parity)

### 📐 等风险贡献 (Equal Risk Contribution)

风险平价由 Qian (2005) 提出，因桥水基金的全天候策略而闻名。核心思想：

> 让投资组合中的每只股票对组合总风险的**贡献相等**。

数学表达：

$$w_i \times \frac{(\Sigma w)_i}{\sigma_p} = \frac{\sigma_p}{N}, \quad \forall i$$

转化为优化问题：

$$\min_{w} \sum_{i=1}^{N} \sum_{j=1}^{N} \left( w_i (\Sigma w)_i - w_j (\Sigma w)_j \right)^2$$
$$\text{s.t.} \quad w'1 = 1, \quad w \geq 0$$

#### 💡 经济含义

- 全天候策略：无论经济环境如何（通胀/通缩、增长/衰退），每只股票的风险贡献相同
- 与 MVO 不同，风险平价**不需要估计预期收益率**
- 通常用于大类资产配置，而非个股选择

In [ ]:
# ========== 风险平价组合 ==========
print("📊 风险平价组合计算")
print("─" * 50)

def risk_parity_objective(w, Sigma):
    """风险平价目标函数: 最小化各资产风险贡献的差异"""
    port_var = w @ Sigma @ w
    port_std = np.sqrt(port_var)
    marginal_risk = Sigma @ w  # Σw
    risk_contrib = w * marginal_risk / port_std  # 每只股票的风险贡献
    # 目标: 所有风险贡献相等 → 最小化方差
    target_rc = port_std / len(w)  # 目标: 每只股票贡献 σ_p/N
    return np.sum((risk_contrib - target_rc) ** 2)

# scipy 求解
constraints_rp = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
bounds_rp = [(0.01, 1)] * N  # 不允许做空

result_rp = optimize.minimize(risk_parity_objective, w0, args=(Sigma,),
                               method='SLSQP', constraints=constraints_rp,
                               bounds=bounds_rp,
                               options={'ftol': 1e-12, 'maxiter': 1000})

w_rp = result_rp.x

print(f"  风险平价权重:")
for i, s in enumerate(stocks):
    print(f"    w*_{s} = {w_rp[i]:.4f}  ({w_rp[i]*100:.1f}%)")
print(f"    权重之和 = {w_rp.sum():.4f}")

# 验证风险贡献
port_var_rp = w_rp @ Sigma @ w_rp
port_std_rp = np.sqrt(port_var_rp)
marginal_risk_rp = Sigma @ w_rp
risk_contrib_rp = w_rp * marginal_risk_rp / port_std_rp

print(f"\n  📊 验证风险贡献:")
for i, s in enumerate(stocks):
    print(f"    {s}: 风险贡献 = {risk_contrib_rp[i]:.4f}  (目标 = {port_std_rp/N:.4f})")
print(f"    风险贡献标准差 = {risk_contrib_rp.std():.6f}  (应接近 0)")

port_ret_rp = w_rp @ mu
port_sr_rp = port_ret_rp / port_std_rp
print(f"\n  📊 风险平价组合特征:")
print(f"    预期收益 = {port_ret_rp:.4f}%")
print(f"    波动率   = {port_std_rp:.4f}%")
print(f"    Sharpe   = {port_sr_rp:.4f}")

In [ ]:
# ========== 可视化：四种策略权重对比 ==========
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

strategies = {
    'MVO': w_mvo_normalized,
    'Min Variance': w_mv,
    'Max Diversification': w_md,
    'Risk Parity': w_rp
}

for idx, (name, w) in enumerate(strategies.items()):
    ax = axes[idx // 2][idx % 2]
    colors = ['#2ecc71' if wi > 0 else '#e74c3c' for wi in w]
    bars = ax.bar(stocks, w * 100, color=colors, edgecolor='black', alpha=0.85)
    for bar, wi in zip(bars, w):
        va = 'bottom' if wi >= 0 else 'top'
        offset = 0.5 if wi >= 0 else -0.5
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + offset,
                f'{wi*100:.1f}%', ha='center', va=va, fontweight='bold', fontsize=9)
    ax.axhline(y=0, color='black', linewidth=0.8)
    ax.set_xlabel('Stock', fontsize=10)
    ax.set_ylabel('Weight (%)', fontsize=10)
    ax.set_title(f'{name} Weights', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 观察：")
print("  - MVO: 偏向高收益股票(A,B)，权重差异最大")
print("  - 最小方差: 偏向低波动率股票(E,D)，更保守")
print("  - 最大多样化: 权重分布较均匀，追求分散化")
print("  - 风险平价: 每只股票风险贡献相等，波动率高的股票权重低")

---

## 6. 四种策略等价条件

### 📖 书中的关键洞察

> 表 7.5 总结了不同配置方法和均值-方差优化的等价条件。

| 方法 | 等价于 MVO 的条件 |
|------|------------------|
| 最小方差 | 所有 $\mu_i$ 相等 |
| 最大多样化 | 所有 $\mu_i/\sigma_i$ 相等 |
| 风险平价 | 所有 $\mu_i/\sigma_i$ 相等 + 所有 $\rho_{ij}$ 相等 |
| 等权重 | 所有 $\mu_i/\sigma_i$ 相等 + 所有 $\rho_{ij}$ 相等 + 所有 $\sigma_i$ 相等 |

等价条件越多，说明该方法对参数假设越苛刻。

### 💡 实验验证

我们设计 4 个实验，逐步放松约束条件，观察各策略权重的差异。

In [ ]:
# ========== 四组实验：逐步放松等价条件 ==========
print("📊 四组实验：不同假设下各策略权重对比")
print("═" * 70)

def run_experiment(mu_exp, sigma_exp, rho_exp, exp_name):
    """在给定参数下运行四种策略"""
    N = len(mu_exp)
    # 构建协方差矩阵
    Sigma_exp = np.outer(sigma_exp, sigma_exp) * rho_exp
    np.fill_diagonal(Sigma_exp, sigma_exp ** 2)
    
    Sigma_inv_exp = np.linalg.inv(Sigma_exp)
    ones_exp = np.ones(N)
    
    # MVO
    w_mvo_exp = Sigma_inv_exp @ mu_exp
    w_mvo_exp = w_mvo_exp / w_mvo_exp.sum()
    
    # 最小方差
    w_mv_exp = Sigma_inv_exp @ ones_exp
    w_mv_exp = w_mv_exp / w_mv_exp.sum()
    
    # 最大多样化
    sigma_vec_exp = sigma_exp
    w_md_exp = Sigma_inv_exp @ sigma_vec_exp
    w_md_exp = w_md_exp / w_md_exp.sum()
    
    # 风险平价
    w0_exp = np.ones(N) / N
    result_exp = optimize.minimize(risk_parity_objective, w0_exp, args=(Sigma_exp,),
                                    method='SLSQP',
                                    constraints=[{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}],
                                    bounds=[(0.01, 1)] * N,
                                    options={'ftol': 1e-12})
    w_rp_exp = result_exp.x
    
    # 等权
    w_ew_exp = np.ones(N) / N
    
    # 计算 Sharpe Ratio
    def calc_sr(w):
        ret = w @ mu_exp
        std = np.sqrt(w @ Sigma_exp @ w)
        return ret / std
    
    return {
        'MVO': w_mvo_exp, 'MinVar': w_mv_exp, 'MaxDiv': w_md_exp,
        'RiskParity': w_rp_exp, 'EW': w_ew_exp,
        'SR_MVO': calc_sr(w_mvo_exp), 'SR_MV': calc_sr(w_mv_exp),
        'SR_MD': calc_sr(w_md_exp), 'SR_RP': calc_sr(w_rp_exp), 'SR_EW': calc_sr(w_ew_exp)
    }

# 实验1: μ/σ 相等 + ρ 相等 + σ 相等 → 等权 ≡ MVO
mu1 = np.array([5.0, 5.0, 5.0])
sigma1 = np.array([10.0, 10.0, 10.0])
rho1 = 0.3
exp1 = run_experiment(mu1, sigma1, rho1, "Exp1")

# 实验2: μ/σ 相等 + ρ 相等 → 风险平价 ≡ MVO
mu2 = np.array([5.0, 7.5, 10.0])
sigma2 = np.array([10.0, 15.0, 20.0])  # μ/σ = 0.5 全部相等
rho2 = 0.3
exp2 = run_experiment(mu2, sigma2, rho2, "Exp2")

# 实验3: μ/σ 相等 → 最大多样化 ≡ MVO
mu3 = np.array([5.0, 7.5, 10.0])
sigma3 = np.array([10.0, 15.0, 20.0])  # μ/σ = 0.5
rho3 = np.array([[1.0, 0.2, 0.4], [0.2, 1.0, 0.3], [0.4, 0.3, 1.0]])  # ρ 不等
exp3 = run_experiment(mu3, sigma3, rho3, "Exp3")

# 实验4: 所有条件都不满足
mu4 = np.array([8.0, 5.0, 3.0])
sigma4 = np.array([15.0, 10.0, 12.0])  # μ/σ 不等
rho4 = np.array([[1.0, 0.2, 0.5], [0.2, 1.0, 0.3], [0.5, 0.3, 1.0]])  # ρ 不等
exp4 = run_experiment(mu4, sigma4, rho4, "Exp4")

# 汇总
labels_exp = ['Exp1: All Equal', 'Exp2: μ/σ+ρ Equal', 'Exp3: μ/σ Equal', 'Exp4: None Equal']
exps = [exp1, exp2, exp3, exp4]

print(f"\n  {'策略':<15} {'Exp1':>10} {'Exp2':>10} {'Exp3':>10} {'Exp4':>10}")
print("  " + "─" * 55)
for strategy in ['MVO', 'MinVar', 'MaxDiv', 'RiskParity', 'EW']:
    row = f"  {strategy:<15}"
    for exp in exps:
        sr_key = f'SR_{strategy.replace("MinVar","MV").replace("MaxDiv","MD").replace("RiskParity","RP")}'
        row += f" {exp[sr_key]:>10.4f}"
    print(row)

print(f"\n  📐 Sharpe Ratio 对比 (MVO 为基准)")
print("  " + "─" * 55)
for i, (exp, label) in enumerate(zip(exps, labels_exp)):
    print(f"  {label}:")
    print(f"    MVO SR = {exp['SR_MVO']:.4f}, 其他策略: EW={exp['SR_EW']:.4f}, "
          f"MV={exp['SR_MV']:.4f}, MD={exp['SR_MD']:.4f}, RP={exp['SR_RP']:.4f}")

print(f"\n  💡 结论:")
print(f"  - 实验1: 所有条件满足 → 所有策略 SR 相同")
print(f"  - 实验2: 放松 σ 条件 → 等权策略 SR 下降")
print(f"  - 实验3: 再放松 ρ 条件 → 风险平价 SR 下降")
print(f"  - 实验4: 所有条件不满足 → 各策略差异最大")

---

## 7. 约束条件

### 📖 书中的七种约束

实际投资组合优化必须配合约束条件，以满足机构或资金的风险偏好。

| 约束类型 | 数学表达 | 经济含义 |
|---------|---------|----------|
| 卖空约束 | $w \geq 0$ | 不允许做空 |
| 杠杆约束 | $w'1 = 1$ | 资金全部使用 |
| 上下限约束 | $L \leq w \leq U$ | 个股权重有上下限 |
| 换手率约束 | $|w - w_0| \leq \phi$ | 控制交易成本 |
| 持仓数量约束 | $N_L \leq \sum \delta_i \leq N_U$ | 控制分散程度 |
| 因子暴露约束 | $l_k \leq w'\beta_k \leq u_k$ | 控制因子暴露 |
| 跟踪误差约束 | $\sqrt{(w-w_B)'\Sigma(w-w_B)} \leq \sigma_{TE}$ | 控制偏离基准 |

### 💡 实际操作演示

我们用 MVO + 各种约束来展示约束的影响。

In [ ]:
# ========== 带约束的 MVO 优化 ==========
print("📊 MVO + 各种约束条件对比")
print("═" * 60)

# 基准参数
mu_c = np.array([8.0, 6.0, 5.0, 4.0, 3.0])
Sigma_c = np.array([
    [16.0,  4.0,  2.0,  1.0,  0.5],
    [ 4.0, 12.0,  3.0,  1.5,  1.0],
    [ 2.0,  3.0, 10.0,  2.0,  1.5],
    [ 1.0,  1.5,  2.0,  9.0,  2.0],
    [ 0.5,  1.0,  1.5,  2.0,  8.0]
])
N_c = len(mu_c)
w0_c = np.ones(N_c) / N_c

def mvo_obj(w, mu, Sigma, zeta):
    return -(w @ mu - zeta / 2 * w @ Sigma @ w)

# 场景1: 无约束 (仅 Σw = 1)
res1 = optimize.minimize(mvo_obj, w0_c, args=(mu_c, Sigma_c, 1.0),
                          method='SLSQP',
                          constraints=[{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}])

# 场景2: 卖空约束 (w >= 0)
res2 = optimize.minimize(mvo_obj, w0_c, args=(mu_c, Sigma_c, 1.0),
                          method='SLSQP',
                          constraints=[{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}],
                          bounds=[(0, 1)] * N_c)

# 场景3: 上下限约束 (每只股票 5%~40%)
res3 = optimize.minimize(mvo_obj, w0_c, args=(mu_c, Sigma_c, 1.0),
                          method='SLSQP',
                          constraints=[{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}],
                          bounds=[(0.05, 0.40)] * N_c)

# 场景4: 换手率约束 (假设当前等权，权重变化不超过 15%)
w_current = np.ones(N_c) / N_c
turnover_limit = 0.15
res4 = optimize.minimize(mvo_obj, w0_c, args=(mu_c, Sigma_c, 1.0),
                          method='SLSQP',
                          constraints=[{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}],
                          bounds=[(max(0, w_current[i] - turnover_limit),
                                   min(1, w_current[i] + turnover_limit)) for i in range(N_c)])

# 场景5: 因子暴露约束 (假设因子1暴露 = [1.2, 0.8, 1.0, 0.9, 1.1])
factor1_exposure = np.array([1.2, 0.8, 1.0, 0.9, 1.1])
# 要求组合在因子1上的暴露在 0.9~1.1 之间（接近中性）
res5 = optimize.minimize(mvo_obj, w0_c, args=(mu_c, Sigma_c, 1.0),
                          method='SLSQP',
                          constraints=[
                              {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
                              {'type': 'ineq', 'fun': lambda w: w @ factor1_exposure - 0.9},
                              {'type': 'ineq', 'fun': lambda w: 1.1 - w @ factor1_exposure}
                          ],
                          bounds=[(0, 1)] * N_c)

# 汇总
scenarios = {
    'Unconstrained': res1,
    'No Short': res2,
    'Bounds [5%,40%]': res3,
    'Turnover <= 15%': res4,
    'Factor Neutral': res5
}

print(f"\n  {'场景':<20} {'A':>8} {'B':>8} {'C':>8} {'D':>8} {'E':>8} {'收益':>8} {'波动率':>8} {'SR':>8}")
print("  " + "─" * 85)

for name, res in scenarios.items():
    w = res.x
    ret = w @ mu_c
    std = np.sqrt(w @ Sigma_c @ w)
    sr = ret / std
    w_str = ' '.join([f'{wi*100:>7.1f}%' for wi in w])
    print(f"  {name:<20} {w_str} {ret:>7.2f}% {std:>7.2f}% {sr:>7.4f}")

print(f"\n  💡 观察:")
print(f"  - 无约束: 可能有负权重(做空)")
print(f"  - 卖空约束: 所有权重 ≥ 0，SR 可能略降")
print(f"  - 上下限约束: 权重被限制在 [5%, 40%]，更加分散")
print(f"  - 换手率约束: 权重变化不超过 15%，偏向等权")
print(f"  - 因子中性: 组合在因子1上的暴露接近 1.0")

In [ ]:
# ========== 可视化约束效果 ==========
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(N_c)
width = 0.15

for i, (name, res) in enumerate(scenarios.items()):
    offset = (i - 2) * width
    bars = ax.bar(x + offset, res.x * 100, width, label=name, alpha=0.85)

ax.set_xlabel('Stock', fontsize=12)
ax.set_ylabel('Weight (%)', fontsize=12)
ax.set_title('MVO Weights Under Different Constraints', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(stocks)
ax.legend(fontsize=9, loc='upper right')
ax.axhline(y=0, color='black', linewidth=0.8)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 约束越强，权重越接近等权组合。极端情况下，强约束 = 退化为等权。")

---

## 8. 交易成本模型

### 📖 书中的交易成本模型

> 交易成本一般分为两种：显性成本和隐性成本，前者包括交易费用和买卖价差，后者主要指的是市场冲击成本。

#### 线性成本函数

$$\text{TC}(w) = \sum_{i=1}^{N} c_i |w_i - w_i^0|$$

其中 $c_i$ 是股票 $i$ 的交易成本比例，$w_i^0$ 是当前权重。

#### 二次成本函数

$$\text{TC}(w) = \sum_{i=1}^{N} c_i (w_i - w_i^0)^2$$

二次成本函数更能反映**市场冲击**：交易量越大，冲击成本越高。

#### 加入目标函数

$$\max_{w} \left[ f(\mu, \Sigma, w) - \gamma_{TC} \cdot \text{TC}(w) \right]$$

其中 $\gamma_{TC}$ 是交易成本的惩罚系数。

In [ ]:
# ========== MVO + 交易成本惩罚 ==========
print("📊 MVO + 交易成本惩罚")
print("═" * 60)

# 当前持仓 (等权)
w_current_tc = np.ones(N_c) / N_c

# 交易成本比例 (假设每只股票 0.3%)
tc_rate = 0.003

def mvo_with_tc(w, mu, Sigma, zeta, w_current, tc_rate, gamma_tc):
    """MVO + 线性交易成本惩罚"""
    ret = w @ mu - zeta / 2 * w @ Sigma @ w
    tc = tc_rate * np.sum(np.abs(w - w_current))
    return -(ret - gamma_tc * tc)

# 不同惩罚系数
gammas = [0, 0.5, 1.0, 2.0, 5.0]

print(f"\n  {'γ_TC':<8} {'A':>8} {'B':>8} {'C':>8} {'D':>8} {'E':>8} {'换手率':>8} {'收益':>8} {'净收益':>8}")
print("  " + "─" * 80)

for gamma in gammas:
    res_tc = optimize.minimize(mvo_with_tc, w0_c,
                                args=(mu_c, Sigma_c, 1.0, w_current_tc, tc_rate, gamma),
                                method='SLSQP',
                                constraints=[{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}],
                                bounds=[(0, 1)] * N_c)
    w_tc = res_tc.x
    turnover = np.sum(np.abs(w_tc - w_current_tc)) / 2
    gross_ret = w_tc @ mu_c
    tc_cost = tc_rate * np.sum(np.abs(w_tc - w_current_tc))
    net_ret = gross_ret - tc_cost
    w_str = ' '.join([f'{wi*100:>7.1f}%' for wi in w_tc])
    print(f"  {gamma:<8.1f} {w_str} {turnover:>7.1%} {gross_ret:>7.2f}% {net_ret:>7.2f}%")

print(f"\n  💡 观察:")
print(f"  - γ_TC = 0: 不考虑交易成本，换手率可能很高")
print(f"  - γ_TC 增大: 换手率下降，权重更接近当前持仓(等权)")
print(f"  - 实际中需要在收益和交易成本之间取舍")

---

## 9. 扩展到完整规模：300 只股票 × 60 月

### 🎯 真实场景模拟

现在我们模拟一个更真实的场景：
- 300 只股票
- 60 个月
- 每月重新优化
- 对比等权 vs MVO vs 最小方差 vs 风险平价

In [ ]:
# ========== 完整规模模拟 ==========
np.random.seed(42)
N_STOCKS_FULL = 50  # 用50只股票模拟(300只太慢)
N_MONTHS_FULL = 60

# 存储各策略的月度收益
strategy_returns = {
    'Equal-Weight': [],
    'MVO': [],
    'Min-Variance': [],
    'Risk-Parity': []
}

print(f"📊 模拟参数:")
print(f"  股票数量: {N_STOCKS_FULL} 只/月")
print(f"  时间跨度: {N_MONTHS_FULL} 个月")
print(f"  策略: 等权, MVO, 最小方差, 风险平价")
print(f"\n🔄 开始逐月优化...")

# 预设因子暴露
factor_size = np.random.randn(N_STOCKS_FULL)  # 市值因子
factor_bm = np.random.randn(N_STOCKS_FULL)    # B/M 因子

for t in range(N_MONTHS_FULL):
    # 生成本月预期收益率
    mu_t = 0.5 + 0.3 * factor_size - 0.2 * factor_bm + np.random.randn(N_STOCKS_FULL) * 2
    
    # 生成协方差矩阵 (简化: 对角 + 少量相关性)
    sigma_t = np.abs(np.random.randn(N_STOCKS_FULL)) * 3 + 5  # 个股波动率 5~14%
    Sigma_t = np.diag(sigma_t ** 2)
    # 添加少量相关性
    for i in range(N_STOCKS_FULL):
        for j in range(i+1, N_STOCKS_FULL):
            corr = 0.3 + np.random.randn() * 0.1
            Sigma_t[i, j] = corr * sigma_t[i] * sigma_t[j]
            Sigma_t[j, i] = Sigma_t[i, j]
    
    # 策略1: 等权
    w_ew_t = np.ones(N_STOCKS_FULL) / N_STOCKS_FULL
    
    # 策略2: MVO (卖空约束)
    w0_t = np.ones(N_STOCKS_FULL) / N_STOCKS_FULL
    try:
        res_mvo_t = optimize.minimize(mvo_obj, w0_t, args=(mu_t, Sigma_t, 1.0),
                                       method='SLSQP',
                                       constraints=[{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}],
                                       bounds=[(0, 0.2)] * N_STOCKS_FULL,
                                       options={'maxiter': 200})
        w_mvo_t = res_mvo_t.x
    except:
        w_mvo_t = w_ew_t
    
    # 策略3: 最小方差
    try:
        res_mv_t = optimize.minimize(min_var_objective, w0_t, args=(Sigma_t,),
                                      method='SLSQP',
                                      constraints=[{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}],
                                      bounds=[(0, 0.2)] * N_STOCKS_FULL,
                                      options={'maxiter': 200})
        w_mv_t = res_mv_t.x
    except:
        w_mv_t = w_ew_t
    
    # 策略4: 风险平价
    try:
        res_rp_t = optimize.minimize(risk_parity_objective, w0_t, args=(Sigma_t,),
                                      method='SLSQP',
                                      constraints=[{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}],
                                      bounds=[(0.001, 0.2)] * N_STOCKS_FULL,
                                      options={'ftol': 1e-10, 'maxiter': 200})
        w_rp_t = res_rp_t.x
    except:
        w_rp_t = w_ew_t
    
    # 计算各策略实际收益 (用本月真实收益率)
    actual_ret = mu_t + np.random.randn(N_STOCKS_FULL) * 3  # 实际收益 = 预期 + 噪声
    
    strategy_returns['Equal-Weight'].append(w_ew_t @ actual_ret)
    strategy_returns['MVO'].append(w_mvo_t @ actual_ret)
    strategy_returns['Min-Variance'].append(w_mv_t @ actual_ret)
    strategy_returns['Risk-Parity'].append(w_rp_t @ actual_ret)
    
    if t < 3:
        print(f"  月份 {t+1}: EW={w_ew_t@actual_ret:.2f}%, MVO={w_mvo_t@actual_ret:.2f}%, "
              f"MV={w_mv_t@actual_ret:.2f}%, RP={w_rp_t@actual_ret:.2f}%")

print(f"\n  ... (省略第 4~{N_MONTHS_FULL} 月) ...")
print(f"✅ 完成！")

In [ ]:
# ========== 策略绩效对比 ==========
print("📊 四种策略绩效对比 (60 个月)")
print("═" * 70)

perf_summary = []
for name, rets in strategy_returns.items():
    rets_arr = np.array(rets)
    mean_ret = rets_arr.mean()
    std_ret = rets_arr.std(ddof=1)
    sr = mean_ret / std_ret * np.sqrt(12)
    cum_ret = np.cumprod(1 + rets_arr / 100) - 1
    max_dd = np.min(np.minimum.accumulate(1 + cum_ret) / np.maximum.accumulate(1 + cum_ret) - 1) * 100
    
    perf_summary.append({
        'Strategy': name,
        'Monthly Mean (%)': mean_ret,
        'Monthly Std (%)': std_ret,
        'Annualized SR': sr,
        'Cumulative (%)': cum_ret[-1] * 100,
        'Max Drawdown (%)': max_dd
    })
    
    # T 检验
    t_stat, p_val = stats.ttest_1samp(rets_arr, 0)
    print(f"\n  {name}:")
    print(f"    月均收益 = {mean_ret:.3f}%, 月波动率 = {std_ret:.3f}%")
    print(f"    年化 SR = {sr:.3f}, 累计收益 = {cum_ret[-1]*100:.1f}%")
    print(f"    T检验: t = {t_stat:.3f}, P = {p_val:.4f}")

df_perf = pd.DataFrame(perf_summary)
print(f"\n📊 绩效汇总表:")
print(df_perf.to_string(index=False))

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- 左图：累计收益曲线 ---
ax1 = axes[0]
colors_strat = {'Equal-Weight': 'steelblue', 'MVO': '#e74c3c',
                'Min-Variance': '#2ecc71', 'Risk-Parity': '#e67e22'}
for name, rets in strategy_returns.items():
    cum = np.cumprod(1 + np.array(rets) / 100)
    ax1.plot(range(1, N_MONTHS_FULL + 1), cum, label=name,
             color=colors_strat[name], linewidth=2)
ax1.set_xlabel('Month', fontsize=12)
ax1.set_ylabel('Cumulative Value (Start = 1)', fontsize=12)
ax1.set_title('Cumulative Performance: 4 Strategies', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# --- 右图：年化 Sharpe Ratio 对比 ---
ax2 = axes[1]
sr_values = [perf['Annualized SR'] for perf in perf_summary]
strat_names = [perf['Strategy'] for perf in perf_summary]
colors_bar = [colors_strat[n] for n in strat_names]
bars = ax2.bar(strat_names, sr_values, color=colors_bar, edgecolor='black', alpha=0.85)
for bar, sr in zip(bars, sr_values):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
             f'{sr:.3f}', ha='center', va='bottom', fontweight='bold')
ax2.set_ylabel('Annualized Sharpe Ratio', fontsize=12)
ax2.set_title('Sharpe Ratio Comparison', fontsize=13, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 观察:")
print("  - 优化策略通常比等权组合有更高的 Sharpe Ratio")
print("  - 最小方差组合波动率最低，但收益也可能较低")
print("  - MVO 收益可能最高，但波动率也大（参数敏感）")
print("  - 风险平价在收益和风险之间取得较好平衡")

---

## 10. 收益与风险模型的错位问题

### 📖 书中的重要警告

> 如果收益率模型中的预测变量和风险模型中构建因子的变量不完全匹配，就会出现收益和风险模型的错位（misalignment）。

### 💡 错位问题的本质

```
收益模型: 用变量 X 预测收益 → R_α = β_α × ω
风险模型: 用变量 Y 计算风险 → R = β_R × λ + ε

如果 X ≠ Y，就会出现错位！

错位的后果:
  β_α 中有一部分不在风险因子的超平面内 → 被误认为"没有风险"
  → 优化器过度倾向于这部分收益 → 实际风险被低估
```

### 解决思路

1. **调整风险模型**：在风险模型中加入收益模型的因子
2. **改进优化过程**：对错位部分的收益进行额外惩罚

$$\max_{w} \left[ w'R_\alpha - \frac{\zeta}{2} w'\Sigma w - \theta \|w'\mu_\perp\|^2 \right]$$

其中 $\theta$ 是非负系数，$\mu_\perp$ 是收益模型中无法被风险因子解释的部分。

---

## 11. 核心概念回顾

### 📌 均值-方差优化 (MVO)
- **定义**: 最大化 $w'\mu - \frac{\zeta}{2} w'\Sigma w$
- **公式**: $w^* \propto \Sigma^{-1}\mu$
- **优点**: 同时考虑收益和风险
- **缺点**: 对 $\mu$ 的估计非常敏感

### 📌 最小方差 (Minimum Variance)
- **定义**: 最小化 $w'\Sigma w$
- **公式**: $w^* \propto \Sigma^{-1}1$
- **优点**: 不需要估计 $\mu$
- **等价条件**: 所有 $\mu_i$ 相等时等价于 MVO

### 📌 最大多样化 (Maximum Diversification)
- **定义**: 最大化 $w'\sigma / \sqrt{w'\Sigma w}$
- **公式**: $w^* \propto \Sigma^{-1}\sigma$
- **等价条件**: 所有 $\mu_i/\sigma_i$ 相等时等价于 MVO

### 📌 风险平价 (Risk Parity)
- **定义**: 每只股票的风险贡献相等
- **公式**: $w_i (\Sigma w)_i / \sigma_p = \sigma_p / N, \forall i$
- **等价条件**: 所有 $\mu_i/\sigma_i$ 相等 + 所有 $\rho_{ij}$ 相等时等价于 MVO

### 📌 约束条件
- **卖空约束**: $w \geq 0$
- **杠杆约束**: $w'1 = 1$
- **上下限约束**: $L \leq w \leq U$
- **换手率约束**: $|w - w_0| \leq \phi$
- **因子暴露约束**: $l_k \leq w'\beta_k \leq u_k$
- **跟踪误差约束**: $\sqrt{(w-w_B)'\Sigma(w-w_B)} \leq \sigma_{TE}$

### 📌 交易成本模型
- **线性**: $\text{TC} = \sum c_i |w_i - w_i^0|$
- **二次**: $\text{TC} = \sum c_i (w_i - w_i^0)^2$
- **加入目标**: $\max [f(\mu, \Sigma, w) - \gamma_{TC} \cdot \text{TC}(w)]$

### 🔗 完整流程
```
收益模型(μ) + 风险模型(Σ) → 选择目标函数 → 设置约束条件
    → 优化求解 → 加入交易成本惩罚 → 最优权重 w*
    → 事后评估(SR, 最大回撤) → 定期再平衡
```

### 📝 策略对比汇总

| 方法 | 需要 μ? | 需要 Σ? | 等价于MVO的条件 | 适用场景 |
|------|---------|---------|----------------|----------|
| MVO | ✅ | ✅ | — | 有可靠的收益预测 |
| 最小方差 | ❌ | ✅ | μ_i 全相等 | 收益预测不可靠 |
| 最大多样化 | ❌ | ✅ | μ_i/σ_i 全相等 | 追求分散化 |
| 风险平价 | ❌ | ✅ | μ_i/σ_i + ρ_ij 全相等 | 大类资产配置 |
| 等权重 | ❌ | ❌ | 三个条件全满足 | 无法估计参数时的无奈之举 |

---

## 12. 常见误区

### ❌ 误区 1: MVO 是最优的
**✓ 正确理解**: MVO 对输入参数（尤其是 μ）非常敏感。参数的微小变化可能导致权重剧烈波动。实际中常需要额外的正则化或约束来稳定结果。

### ❌ 误区 2: 等权组合是"最优"的懒人策略
**✓ 正确理解**: 等权组合只有在三个苛刻条件（μ/σ 相等、ρ 相等、σ 相等）同时满足时才等价于 MVO。一旦有任何参数信息，等权就不是最优的。

### ❌ 误区 3: 只关注目标函数，忽视约束条件
**✓ 正确理解**: 实际投资中，约束条件（卖空、换手率、因子暴露等）对最终权重的影响可能比目标函数更大。选错约束 = 优化结果不可执行。

### ❌ 误区 4: 忽视收益和风险模型的错位
**✓ 正确理解**: 如果收益模型和风险模型使用的变量不一致，优化器会低估某些收益的风险，导致权重偏向这些"虚假安全"的收益。国内很多券商报告忽视了这个问题。

### ❌ 误区 5: 认为优化一次就够了
**✓ 正确理解**: 投资组合优化需要**定期再平衡**。市场变化、因子暴露漂移、交易成本累积都要求动态调整。频率太高（交易成本高）和太低（因子暴露漂移）都需要权衡。